# Positional Encoding

Construct sinusoidal position vectors, add them to token embeddings, and compare with a learned position table. Small dimensions make every value visible.

In [ ]:
import math
import torch
from torch import nn

torch.set_num_threads(1)
print('CPU | position vectors')

In [ ]:
def sinusoidal_positions(length, width):
    positions = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    frequencies = torch.exp(torch.arange(0, width, 2, dtype=torch.float32) * (-math.log(10000.0) / width))
    encoding = torch.zeros(length, width)
    encoding[:, 0::2] = torch.sin(positions * frequencies)
    encoding[:, 1::2] = torch.cos(positions * frequencies)
    return encoding

encoding = sinusoidal_positions(3, 4)
expected_position_one = torch.tensor([math.sin(1), math.cos(1), math.sin(0.01), math.cos(0.01)])
torch.testing.assert_close(encoding[1], expected_position_one)
assert encoding.shape == (3, 4)
print('position 0:', encoding[0].tolist())
print('position 1:', encoding[1].tolist())

In [ ]:
tokens = torch.tensor([[2, 5, 2]])
token_embedding = nn.Embedding(10, 4)
token_vectors = token_embedding(tokens)
position_vectors = sinusoidal_positions(tokens.shape[1], 4).unsqueeze(0)
combined = token_vectors + position_vectors
assert combined.shape == (1, 3, 4)
assert not torch.allclose(combined[:, 0], combined[:, 2])
learned_positions = nn.Embedding(8, 4)(torch.arange(3))
print('token vectors:', tuple(token_vectors.shape), '| combined:', tuple(combined.shape))
print('learned position table sample:', learned_positions[0].tolist())